In [9]:
import pandas as pd

In [15]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

# Load Helsinki model
tokenizer_h = AutoTokenizer.from_pretrained("Helsinki-NLP/opus-tatoeba-en-tr")
model_h = AutoModelForSeq2SeqLM.from_pretrained("Helsinki-NLP/opus-tatoeba-en-tr")

/usr/local/lib/python3.12/dist-packages/transformers/models/marian/tokenization_marian.py:175: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


In [16]:
from transformers import M2M100ForConditionalGeneration, M2M100Tokenizer

model_m2 = M2M100ForConditionalGeneration.from_pretrained("facebook/m2m100_418M")
tokenizer_m2 = M2M100Tokenizer.from_pretrained("facebook/m2m100_418M")

In [17]:
model_configs = [
    #{"target_column": "model1", "tokenizer": tokenizer_h, "model": model_h, "src_lang": "eng", "tgt_lang": "tur"},
    {"target_column": "model2", "tokenizer": tokenizer_m2, "model": model_m2, "src_lang": "en", "tgt_lang": "tr"},
    #{"target_column": "model3", "tokenizer": tokenizer_m, "model": model_m, "src_lang": "en", "tgt_lang": "tr"},
    #{"target_column": "model4", "tokenizer": tokenizer_n, "model": model_n, "src_lang": "eng_Latn", "tgt_lang": "tur_Latn"},
]

In [18]:
# Function to fill a model column with translations for Helsinki
def fill_model_column_h(df, source_column, target_column, tokenizer, model, batch_size=32):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model = model.to(device)
    sentences = df[source_column].tolist()
    translations = []

    # Batch translation for efficiency
    for i in range(0, len(sentences), batch_size):
        batch = sentences[i:i+batch_size]
        inputs = tokenizer(batch, return_tensors="pt", padding=True, truncation=True).to(device)
        with torch.no_grad():
            outputs = model.generate(**inputs)
        decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)
        translations.extend(decoded)
    
    # Fill the target column
    df[target_column] = translations
    return df

In [19]:
def fill_model_column(df, source_column, target_column, src_lang, tgt_lang, tokenizer, model, batch_size=32):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model = model.to(device)
    tokenizer.src_lang = src_lang
    sentences = df[source_column].tolist()
    translations = []

    for i in range(0, len(sentences), batch_size):
        batch = sentences[i:i+batch_size]

        encoded = tokenizer(
            batch,
            return_tensors="pt",
            padding=True,
            truncation=True
        ).to(device)

        with torch.no_grad():
            generated_tokens = model.generate(
                **encoded,
                forced_bos_token_id=tokenizer.lang_code_to_id[tgt_lang],
                max_length=128
            )

        decoded = tokenizer.batch_decode(generated_tokens, skip_special_tokens=True)
        translations.extend(decoded)

    df[target_column] = translations
    return df

In [21]:
df3_translated = pd.read_csv("/kaggle/input/df3-sp-done-new/df3_sp_done_new.csv")

In [22]:
# Fill model1 column for df2
df3_translated = fill_model_column_h(df3_translated, source_column="source_sentence", target_column="model1",
                            tokenizer=tokenizer_h, model=model_h,batch_size=32)

#df2.to_csv("data_nmt_2_with_model2.csv", index=False)

In [23]:
for config in model_configs:
    print(f"Translating for column: {config['target_column']}")
    df3_translated = fill_model_column(
        df=df3_translated,
        source_column="source_sentence",
        target_column=config["target_column"],
        src_lang=config["src_lang"],
        tgt_lang=config["tgt_lang"],
        tokenizer=config["tokenizer"],
        model=config["model"],
        batch_size=32
    )

df3_translated.to_csv("data_nmt_3.csv", index=False)

Translating for column: model2
